# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [1]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-05-31 01:58:58,226 root INFO: Loading model from k2-fsa/OmniVoice, device=cuda ...
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-31 01:58:58,539 huggingface_hub.utils._http WARNING: Warning: You are se

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

In [ ]:
audio_chinese = model.generate(
    text="打打嗝说中文。",
)

sf.write("chinese_out.wav", audio_chinese[0], 24000)
display(Audio(audio_chinese[0], rate=24000))

In [ ]:
audio_sichuanese = model.generate(
    text="打打嗝说中文。",
    instruct="male, old, extremely low, sichuan dialect"
)

sf.write("sichuanese_out.wav", audio_sichuanese[0], 24000)
display(Audio(audio_sichuanese[0], rate=24000))

In [ ]:
long_chinese_text = """天不生仲尼，万古如长夜。然而，即便天降仲尼，一个三岁丧父、在颜母施氏一手抚养下成长于阙里的小孩，其恢弘的哲学体系，根源究竟深植何处？若我们引入一个比喻性的概念——“文化染色体”，那么孔子的思想，无疑拥有着令人惊叹的 “端粒”长度——一种超乎寻常的遗传稳定性和延续力。直至今日，若将中华文明的核心基因视作一套染色体组，那么孔子，依然是当之无愧的 “一号染色体” ，其后任何思想家，都难逃其巨大的身影，唯有高山仰止。那么，这位万世开太平的至圣先师，其学问究竟师从何人？答案，或许并不在鲁国的典籍库中，而在于一个更为古老、更为磅礴的源头——昆仑。"""

audio_confucius = model.generate(
    text=long_chinese_text,
    instruct="male, old, extremely low, sichuan dialect"
)

sf.write("confucius_sichuanese_out.wav", audio_confucius[0], 24000)
display(Audio(audio_confucius[0], rate=24000))

## 制作播客：第六章 孔子和麒麟

我们将把您提供的文本分解成几部分，并用“男性、老年、极低、四川话”的组合来生成音频，模拟播客效果。

In [ ]:
podcast_segments = {
    "intro": "引言：文化染色体与至圣先师\n天不生仲尼，万古如长夜。然而，即便天降仲尼，一个三岁丧父、在颜母施氏一手抚养下成长于阙里的小孩，其恢弘的哲学体系，根源究竟深植何处？若我们引入一个比喻性的概念——“文化染色体”，那么孔子的思想，无疑拥有着令人惊叹的 “端粒”长度——一种超乎寻常的遗传稳定性和延续力。直至今日，若将中华文明的核心基因视作一套染色体组，那么孔子，依然是当之无愧的 “一号染色体” ，其后任何思想家，都难逃其巨大的身影，唯有高山仰止。\n那么，这位万世开太平的至圣先师，其学问究竟师从何人？答案，或许并不在鲁国的典籍库中，而在于一个更为古老、更为磅礴的源头——昆仑。",
    "section1": "第一节 巨野的瑞兽：麟\n巨野流传着不少和孔子相关的传说，最核心的是 “孔母梦麟”。这是传说颜徵在随家人从祖籍河南商丘返回曲阜途中，路过巨野麟山歇脚时，梦见麒麟入怀，之后便怀了孔子——顺便说一下，那位国军将领蒋梦麟的名字即源于此。此外，巨野还有 “西狩获麟” 的典故，说的是鲁哀公在巨野的大野泽附近猎杀了麒麟，孔子认出后悲痛不已，从此《春秋》停笔。\n麒麟送子的源头就是孔母颜徵。但吊诡的是：为什么很多人都知道孔母梦麟，但是不知道“麒麟送子”这个词，居然是源于孔子？事情的原委，恐怕比我们一般人预料的要复杂。\n龙为长虫，麟为走兽，麟者，龙也。至于鹿这个偏旁——还记得我们之前关于象雄文明，鄂霍次克文明乃至《蒙古秘史》中对白鹿的推崇吗？没错，鹿就是路，道也。而给麟表音的粦，则是桀这个字的上下结构的反转。粦为阳，为祥；桀为阴，为病。\n那么我为什么非得这么解释呢？因为，在甲骨文当中居然不存在——桀，这个字！\n《今文尚书》中，《汤誓》和《牧誓》是姊妹篇，一个是商汤要灭夏桀，一个是周武要灭商纣，互文如双鱼玉佩。\n内蒙古通辽契丹古墓双鱼玉佩\n然而，还是那句话，甲骨文中并没有——桀。",
    "section2": "第二节 万世师表 地水为师\n地为坤德，厚德载物；水为坎象，险陷重重。然而，地与水相叠，却成就了《易经》中唯一象征“武之极征服”与“文之巅教化”的卦象——师卦，武曰正义之师、文颂帝王之师。然而孔子弟子却称呼自己为：夫子。\n怪吗？但这恰如孔子的一生：他行走于坚实的大地（现实人间），深入纷繁复杂的世情（坎水之险），将个体的阅历与感悟，汇聚成滋养天下的智慧渊泉，终至“容保民无疆”的师表境界。\n孔子在《易传·象传》中为师卦写下的断语，正是其教育哲学与政治理想的终极写照：\n这短短十字，道尽万世师心的精髓：\n“容民” 是胸怀，是如大地般宽厚的包容与共情（“有教无类”）。\n“畜众” 是方法，是如水汇于渊薮般的教化与滋养（“诲人不倦”）。\n而他为师卦卦辞所作的深邃阐发，更将武力之师升华为教化之师：\n此言如黄钟大吕，道出权力的真谛：统领千军万马（“师，众也”）只是表象，能引导大众归于正道（“以众正”），才堪为真正的王者（“可以王矣”）。\n这正是他从“地水师”的卦象中，解读出的万世不易的“王道”——教育，才是最高的政治。\n孔子并不是书斋里的经师，而是一位行走在时代刀锋上的“深度调查记者”。其的思想体系，并非来自静态的苦读或天启，而是源于一场持续一生的、极其艰苦且自觉的 “社会田野调查”。\n方法论：用脚步丈量的“活”的学问\n田野现场：他的书房是整个春秋列国。周游十四年的颠沛流离，不是失意政客的流浪，而是首席记者带着团队（弟子们）进行的覆盖多国的深度调研。各国政要、隐士、农夫都是他的采访对象。
核心信源：他“入太庙，每事问”，对《诗》《书》等古籍的钻研，不是死记硬背，而是核查事实、比对史料的严谨过程。他是在与活生生的现实和古老文献的双重对话中，寻找真相。
驱动力：“天生天养”塑造的独立视角
“局外人”的优势：父亲早逝，大家族的荫蔽缺失，使他无法轻易接受现成的世界解释。这迫使他必须独自上路，为自己也为时代寻找答案。
这种“野生”的生存状态，塑造了他不依附任何权威的、批判性的“局外人”视角——这正是顶级调查记者的核心品质。
终极选题：他所处的“礼崩乐坏”时代，就是他穷尽一生要破解的 “核心报道选题”：
一个文明为何会崩溃？
重建的基石又在何处？
成果：从“调查”到“思想体系”的构建
他将调查所得的海量信息（各国政情、民风、古籍），用 “仁”与“礼” 这一套强大的分析框架进行整合。
“仁” 是他洞察到的人性内核与社会运行的道德基石（相当于报道的“中心思想”）。
“礼” 是他找到的维系社会和谐的具体规则与秩序（相当于报道提出的“解决方案”）。
因此，孔子这位“万世师表”，其表率之处，正是这“地水”之德：
厚德载物，何以载德？以德治国，文以载道！",
    "section3": "第三节 克己复礼 谁的礼？\n据《春秋纬・演孔图》记载：\n这一段谶纬，常被视为荒诞不经的神话，但在我们的“文明审计”中，恰似一把密钥，精诚所至，金石为开。\n黑帝者，颛顼也，镇北之黑水。\n但这段话的重点不在“梦交”，而在“空桑”——翻开上古神话谱系，还有谁生于空桑？\n伊尹！伊尹者，一一也。子就是第一，孔子如果按照原名就是子子，也是一一！有那味儿了！\n《吕氏春秋・本味》\n相传伊尹之母梦见大水将至，化为空桑（空心的桑树），伊尹便生于其中。伊尹是谁？恰似姜尚灭商的角色，商汤灭夏的总设计师就是伊尹，是所谓“元圣”。\n《演孔图》把孔子的出生地定在“空桑”，这绝非巧合，而是一次隐秘的“政治认祖”。 这意味着，在汉代知情者的潜意识里，孔子拿的剧本，根本不是周公的剧本，而是伊尹或者姜尚的剧本。 他不是来维护周朝统治的，他是来“借壳上市”的——借周礼的壳，反周而复商之礼。\n礼失，需求诸于野，于是孔子在52岁高龄硬是又游了14年。\n当他在太庙“每事问”的过程中，不可能不刨根问底，比如在对比《汤誓》与《牧誓》这块“双鱼玉佩”时，有没有可能发现了我们前面的问题：\n那些不需土刨，俯仰皆是的甲骨文中里，居然没有“桀”？何故？\n如果夏桀是被周人为了论证“商汤伐桀 = 武王伐纣”的合法性，而进行镜像伪造的产物（桀 = 反转的粦 = 坏掉的祥瑞），那么周朝建立的法理基础（天命转移），本质上就是建立在谎言之上的。\n那么一个建立在谎言之上的“周礼”，怎么可能挽救礼崩乐坏的乱世？\n孔子除了对周武王等第一代领导集体有过称赞之外，对其他周天子无感。从周灭商到秦一统，煌煌八百年，孔子之外，可曾有其他学者说过除了周的第一代领导集体的任何好话？而今二里头突然被立为“夏都”，毋宁说是“夏墟”！\n孔子一生都在使用君子小人来教育他的72门徒，但可曾指名道姓过谁是君子谁是小人吗？\n让我们来大胆猜测一下，孔子认为自己是君子，而周天子，除了第1代领导集体之外，都是小人。\n根据何在？孔子生活在鲁国，周公的封地，周礼的大本营——而他穿的是什么？\n章甫者，丈夫也，而其正是殷商的礼帽。\n在那个“衣冠即政治”的年代，孔子戴着前朝的帽子招摇过市，这不仅是怀旧，而是无声的抗议。\n诸君看图，左图为獬豸冠，中图为甘肃博物馆的青铜獬豸，右图为周润发《孔子》剧照，其形制皆为一根玉簪固定头发的礼器也。何故？獬豸者，一柱擎天，真相不二。说一不二者，真丈夫也。\n我们不妨再重复一次之前的论断，獬豸即麒麟。\n《说文解字》载獬豸 “性忠，见人斗则触不直者，闻人论则咋不正者”，核心是 “辨是非、守正道”；\n《公羊传・哀公十四年》注麒麟 “仁兽也，有王者则至，无王者则不至”，核心是 “应圣世、表仁德”。\n孔子因梦麟而生，十五有志于学，三十而立，四十而不惑，五十而知天命，六十而耳顺，七十而从心所欲，但在他七十二岁的时候，得知麟被射，因获麟而绝笔，七十三岁寿终正寝。而在说“三十而立”这段话的时候，就是在麒麟被射杀的同年——这就是孔子口述的墓志铭。
礼成，麒麟送夫子而来；道穷，夫子陪麒麟而去。
至于孔姓，则是其祖父将其子姓，加了个竖弯钩，训读为孔。 孔者，瞳孔也。不睁开眼睛当然会“万古如长夜”！而那些睁眼瞎更是一叶障目，如同白内障发作。
瞳孔，这个最小之圆是我们认识世界的唯一通路，我们的圣人又何尝不是一孔之见呢？\n孔者，光来之路也。 丘者，岳上之山也。\n我国处于并将长期处于社会主义初级阶段，因为我们生活并将长期生活在孔子之内，不逾矩。
孔丘者，昆仑之道也。",
    "postscript": "后记：南孔北孔——鄂霍次克的回响\n在孔子死后的一千五百年，宋室南渡，孔氏家族也迎来了一次命运的细胞分裂——“孔末乱孔”与“南宗北宗”。 也就是我们俗称的南孔（衢州）与北孔（曲阜）。\n若我们对此进行一次慎重的基因审计，会发现一个极具历史张力的悖论： 身在江南的南孔，或许在血统上，比北孔更“北”。\n现代分子人类学的数据隐晦地指向：南孔的核心单倍群为 C-F1319（或相关C系分支）。 C系是什么？ 正如我们前文所论，那是肃慎，是通古斯，是鄂霍次克海沿岸那凛冽寒风中走出的古老猎人，是殷商王族最纯正的底色。 孔子作为殷商微子启的后裔，他身上流淌的，本就是来自白山黑水的血液。\n历史仿佛开了一个玩笑： 当金兵南下，孔子最嫡系的血脉（衍圣公孔端友）背负着那对传家宝——楷木像（孔子夫妇像），渡江南下，隐入衢州的烂柯山旁。 他们把“鄂霍次克-殷商”那条最古老的C系染色体，带到了温暖湿润的江南，像一颗种子一样封存了起来（冷备份）。\n而留在北方的北孔，在金、元、明、清的更迭中，更多承担了“守陵”与“祭祀”的政治功能（热数据）。在漫长的战乱与融合中，北方的基因图谱不可避免地发生了更多元（O系融入）的重构。
但这并不影响孔子的伟大。 相反，这构成了一种完美的双活架构（Dual-Active Architecture）：\n北孔（曲阜）： 守住了“土”。那是物理上的昆仑，是礼教的道场，接受万世的朝拜。
南孔（衢州）： 守住了“血”。那是生物学上的昆仑，是鄂霍次克海的古老记忆，在江南的烟雨中，静静延续着那个关于“玄鸟生商”的基因神话。\n无论南北，皆是圣裔。 只是当我们凝视南孔后人的脸庞时，或许能透过那江南的温润，依稀看到一丝来自远古北亚的坚毅轮廓——那是孔夫子真正的“出厂设置”。"
}

# 定义男声和女声的指令参数
male_instruct_params = "male, old, extremely low, sichuan dialect"
female_instruct_params = "female, middle age, medium pitch"

# 为每个片段指定朗读者的指令参数
segment_voice_mapping = {
    "intro": female_instruct_params,
    "section1": male_instruct_params,
    "section2": male_instruct_params,
    "section3": male_instruct_params,
    "postscript": female_instruct_params
}

# Generate audio for each segment
for key, text in podcast_segments.items():
    print(f"Generating audio for: {key}")
    current_instruct_params = segment_voice_mapping.get(key, male_instruct_params) # Default to male if not specified
    audio_output = model.generate(
        text=text,
        instruct=current_instruct_params
    )
    filename = f"podcast_{key}.wav"
    sf.write(filename, audio_output[0], 24000)
    display(Audio(audio_output[0], rate=24000))

## 组合播客片段 (可选)

如果您想将这些单独的音频片段组合成一个完整的播客文件，您可以使用 `pydub` 库。这需要先安装 `pydub` 和 `ffmpeg`。

In [ ]:
#!pip install pydub
# import pydub
# from pydub import AudioSegment

# # Ensure ffmpeg is installed, for example:
# # !apt-get update && apt-get install -y ffmpeg

# combined_podcast = AudioSegment.empty()

# for key in podcast_segments.keys():
#     filename = f"podcast_{key}.wav"
#     segment = AudioSegment.from_wav(filename)
#     combined_podcast += segment
#     # Optional: add a small pause between segments
#     # combined_podcast += AudioSegment.silent(duration=500) # 500ms pause

# combined_podcast.export("full_podcast_confucius.mp3", format="mp3")
# print("Full podcast combined and saved as full_podcast_confucius.mp3")